
# Planet PSScene 2020 — AOI Search → Batched Orders (clip → reproject → COG) → Download

This notebook automates:
1) Reading your AOI (GeoJSON).  
2) Searching **PSScene (PlanetScope)** scenes in **2020** with a cloud filter.  
3) Placing **batched Orders** with a **toolchain**: `clip → reproject → COG`.  
4) Downloading into a tidy local folder structure you can feed into your labeling / mask-rasterization pipeline.

> **Auth:** Run `planet auth login` once so the SDK can pick it up.  
> **Quota:** Running orders consumes quota; double‑check filters before executing.


In [1]:
# %pip install -q planet geopandas shapely tqdm pandas

import os
import json
from pathlib import Path
from datetime import datetime, date

from tqdm import tqdm
import pandas as pd

from planet import Planet, data_filter, order_request, reporting

try:
    import geopandas as gpd  # optional
except Exception:
    gpd = None

AOI_DIR = Path(r"D:\Thesis\glacial_lake_detection_thesis\data\raw\geojson\inference\all_leftover")   # set this to your folder path
GEOJSON_PATTERN = "*.geojson"
OUT_DIR = Path(r"D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\inference\planet_downloads_PSScene_rgb_2025")
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2025
DATE_START = f"{YEAR}-09-16"
DATE_END = f"{YEAR}-09-17"
MAX_CLOUD = 0.2

PRIMARY_BUNDLE = "visual"
#PRIMARY_BUNDLE = "analytic_8b_sr_udm2"
FALLBACK_BUNDLES = ["analytic_sr_udm2", "analytic_8b_udm2"]

REPROJECT_EPSG = "EPSG:4326"
MAX_ITEMS_PER_ORDER = 400
WAIT_MAX_ATTEMPTS = 600
ZIP_EACH_ORDER = False


In [2]:
!planet auth login
pl = Planet()
print("Planet client ready. If this fails, run `!planet auth login`.")


Logging in with authentication profile planet-user...
Opening browser to login.
Confirm the authorization code when prompted: DWBC-TMVV

Login succeeded.
Planet client ready. If this fails, run `!planet auth login`.


In [3]:
def read_aoi_geojson(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"AOI GeoJSON not found: {path}")
    obj = json.loads(path.read_text())
    if obj.get("type") == "FeatureCollection":
        geom = obj["features"][0]["geometry"]
    elif obj.get("type") == "Feature":
        geom = obj["geometry"]
    else:
        geom = obj
    if geom.get("type") not in {"Polygon", "MultiPolygon"}:
        raise ValueError("AOI must be a Polygon or MultiPolygon geometry.")
    return geom


In [4]:
def process_one_aoi(aoi_path: str|Path):
    """Run the same single-AOI flow for a given GeoJSON path.
    Creates a subfolder under OUT_DIR using the AOI filename stem.
    Returns list of created order IDs.
    """
    
    aoi_path = Path(aoi_path)
    aoi_geom = read_aoi_geojson(aoi_path)

    # Ensure sub-output dir per AOI
    aoi_name = aoi_path.stem
    aoi_out = OUT_DIR / aoi_name
    aoi_out.mkdir(parents=True, exist_ok=True)

    # ---- Search (mirrors earlier search cell) ----
    items = []
    sr_or = data_filter.or_filter([
    data_filter.asset_filter(asset_types=["ortho_visual"]),  # 8-band SR "ortho_analytic_8b_sr"
    #data_filter.asset_filter(asset_types=["ortho_analytic_sr"]),     # 4-band SR
    ])

    udm2_or = data_filter.or_filter([
        data_filter.asset_filter(asset_types=["ortho_analytic_udm2"]),
        data_filter.asset_filter(asset_types=["ortho_udm2"]),            # some items expose this
    ])
    sfilter = data_filter.and_filter([
        data_filter.std_quality_filter(),
        sr_or,
        udm2_or,
        data_filter.date_range_filter("acquired", gte=datetime.fromisoformat(DATE_START), lte=datetime.fromisoformat(DATE_END)),
        data_filter.range_filter("cloud_cover", lte=MAX_CLOUD),
    ])

    for item in pl.data.search(["PSScene"], geometry=aoi_geom, search_filter=sfilter, limit=10000):
        props = item.get("properties", {})
        items.append({
            "id": item.get("id"),
            "acquired": props.get("acquired"),
            "cloud_cover": props.get("cloud_cover"),
            "satellite_id": props.get("satellite_id"),
            "strip_id": props.get("strip_id"),
            "publishing_stage": props.get("publishing_stage"),
        })

    df = pd.DataFrame(items).sort_values(["cloud_cover", "acquired"])
    print(f"[{aoi_name}] Found {len(df)} items.")

    # Save CSV per AOI
    csv_path = aoi_out / f"search_results_{YEAR}_{aoi_name}.csv"
    df.to_csv(csv_path, index=False)
    print(f"[{aoi_name}] Wrote: {csv_path}")

    # ---- Chunk into batches ----
    def chunk(lst, n):
        for i in range(0, len(lst), n):
            yield lst[i:i+n]
    item_ids = df["id"].tolist()
    batches = list(chunk(item_ids, MAX_ITEMS_PER_ORDER))
    print(f"[{aoi_name}] Total items: {len(item_ids)} across {len(batches)} order(s).")
    return batches


In [24]:

# === Quota charge estimate (per-scene & per-order) ===
# Computes how much area (km²) Planet will deduct from your quota for this order,
# using Planet's Preferred vs Premium charging rules.
# References:
# - Preferred vs Premium minimums: 100 km² per intersecting scene vs actual area. See Orders API FAQ. 
# - New Quota Reservations API can also estimate quota for supported products.
# Docs: https://support.planet.com/hc/en-us/articles/360011216618-Orders-API-FAQ
#       https://docs.planet.com/develop/apis/quota/
from shapely.geometry import shape
from shapely.ops import transform
import pyproj
import pandas as pd
from tqdm import tqdm

PLAN_MODEL = globals().get("PLAN_MODEL", "preferred").lower()  # 'preferred' (E&R Basic) or 'premium'
EQUAL_AREA_CRS = "EPSG:6933"  # World Cylindrical Equal Area



def fetch_item_geometry(item_id: str):
    """Fetch PSScene item geometry via SDK; fallback to Data API request if needed."""
    try:
        item = pl.data.get_item("PSScene", item_id)  # SDK call (sync helper in this notebook's Planet wrapper)
        geom = item.get("geometry") or item.get("_links", {}).get("_self_geometry")  # robust-ish
        if geom:
            return shape(geom)
    except Exception as e:
        pass
    # Fallback to raw HTTP if SDK isn't available
    import os, requests, base64
    api_key = os.getenv("PL_API_KEY") or os.getenv("PL_APIKEY") or os.getenv("PL_API_KEY_ID") or os.getenv("PL_APIKEY_ID")
    if not api_key:
        # try reading the planet auth file written by `planet auth login`
        try:
            import json, pathlib
            auth_path = pathlib.Path.home()/".planet.json"
            if auth_path.exists():
                api_key = json.loads(auth_path.read_text()).get("key")
        except Exception:
            api_key = None
    if not api_key:
        raise RuntimeError("Planet API key not found. Set PL_API_KEY env var or run `!planet auth login`.")
    url = f"https://api.planet.com/data/v1/item-types/PSScene/items/{item_id}"
    resp = requests.get(url, auth=(api_key, ""))
    resp.raise_for_status()
    return shape(resp.json()["geometry"])

def estimate_for_ids(ids: list[str]) -> pd.DataFrame:
    rows = []
    for sid in tqdm(ids, desc="Fetching footprints & intersecting AOI"):
        try:
            scene_geom = fetch_item_geometry(sid)
        except Exception as e:
            # If footprint not available, assume conservative minimum charge if Preferred; else 0
            rows.append({
                "scene_id": sid,
                "intersect_km2": float('nan'),
                "charged_km2": 100.0 if PLAN_MODEL.startswith("pref") else float('nan'),
                "note": "geometry fetch failed; using minimum for Preferred" if PLAN_MODEL.startswith("pref") else "geometry fetch failed"
            })
            continue
        scene_ea = transform(to_ea, scene_geom)
        inter = aoi_ea.intersection(scene_ea)
        inter_area_km2 = inter.area / 1e6  # m² to km²

        if PLAN_MODEL.startswith("pref"):
            charged = max(100.0, inter_area_km2) if inter_area_km2 > 0 else 0.0
        else:
            charged = inter_area_km2

        rows.append({
            "scene_id": sid,
            "intersect_km2": round(inter_area_km2, 3),
            "charged_km2": round(charged, 3),
            "note": ""
        })
    df_est = pd.DataFrame(rows)
    if not df_est.empty:
        total = df_est["charged_km2"].fillna(0).sum()
        print(f"Estimated quota charge for {len(ids)} scene(s) under plan='{PLAN_MODEL}': {total:.2f} km²")
    return df_est

# Prepare transformers
to_ea = pyproj.Transformer.from_crs("EPSG:4326", EQUAL_AREA_CRS, always_xy=True).transform
aoi_files = sorted(AOI_DIR.glob(GEOJSON_PATTERN))
print(f"Batch AOIs in {AOI_DIR} → {len(aoi_files)} files")
if not aoi_files:
    raise FileNotFoundError(f"No GeoJSON files matched: {AOI_DIR / GEOJSON_PATTERN}")

all_results = {}
for i, aoi in enumerate(aoi_files, start=1):
    print(f"\n=== [{i}/{len(aoi_files)}] {aoi.name} ===")
    aoi_geom = read_aoi_geojson(aoi)
    aoi_poly = shape(aoi_geom)
    aoi_ea = transform(to_ea, aoi_poly)

    # Compute per-batch and total estimates BEFORE creating orders
    all_rows = []
    batches = process_one_aoi(aoi)
    for idx, batch_ids in enumerate(batches, start=1):
        print(f"\n=== Estimating order {idx}/{len(batches)} ({len(batch_ids)} items) ===")
        df_batch = estimate_for_ids(batch_ids)
        df_batch["order_idx"] = idx
        order_total = df_batch["charged_km2"].fillna(0).sum()
        print(f"Order {idx} subtotal (km²): {order_total:.2f}")
        all_rows.append(df_batch)

    quota_estimates = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
    quota_estimates["aoi_path"] = aoi
    display_cols = ["order_idx", "scene_id", "intersect_km2", "charged_km2", "note", "aoi_path"]
    quota_estimates = quota_estimates[display_cols] #.sort_values(["order_idx", "charged_km2"], ascending=[True, False])
    aoi_name = aoi.stem
    aoi_out = OUT_DIR / aoi_name
    csv_path = aoi_out / f"quota_{YEAR}_{aoi_name}.csv"
    quota_estimates.to_csv(csv_path, index=False)
    print(quota_estimates.head(20))


Batch AOIs in D:\Thesis\glacial_lake_detection_thesis\data\raw\geojson\inference\aoi40 → 1 files

=== [1/1] aoi40.geojson ===
[aoi40] Found 4 items.
[aoi40] Wrote: D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\inference\planet_downloads_PSScene_rgb_2025\aoi40\search_results_2025_aoi40.csv
[aoi40] Total items: 4 across 1 order(s).

=== Estimating order 1/1 (4 items) ===


Fetching footprints & intersecting AOI: 100%|██████████| 4/4 [00:01<00:00,  2.78it/s]

Estimated quota charge for 4 scene(s) under plan='preferred': 795.95 km²
Order 1 subtotal (km²): 795.95
   order_idx                 scene_id  intersect_km2  charged_km2 note  \
0          1  20250916_062858_12_2508         88.258      100.000        
1          1  20250916_062900_26_2508        303.652      303.652        
2          1  20250916_062902_40_2508        292.301      292.301        
3          1  20250916_062904_55_2508         48.756      100.000        

                                            aoi_path  
0  D:\Thesis\glacial_lake_detection_thesis\data\r...  
1  D:\Thesis\glacial_lake_detection_thesis\data\r...  
2  D:\Thesis\glacial_lake_detection_thesis\data\r...  
3  D:\Thesis\glacial_lake_detection_thesis\data\r...  


In [26]:
aoi_files = sorted(AOI_DIR.glob(GEOJSON_PATTERN))
path_list = []
for aoi_file in aoi_files:
    aoi_name = aoi_file.stem
    aoi_out = OUT_DIR / aoi_name
    csv_path = aoi_out / f"quota_{YEAR}_{aoi_name}.csv"
    path_list.append(csv_path)
frames = [pd.read_csv(p, low_memory=False).assign(source_file=p.name) for p in path_list]

# concatenate into a single DataFrame
df = pd.concat(frames, ignore_index=True, sort=False)
df.to_csv(OUT_DIR / "quota_concatenated.csv", index=False)

In [7]:
df = pd.read_csv(OUT_DIR / "quota_concatenated.csv")
order_df = df.tail(20)
order_df


,order_idx,scene_id,intersect_km2,charged_km2,note,aoi_path,source_file
5,1,20250913_061703_50_2547,258.389,258.389,NaN,D:\Thesis\glacial_lake_detection_thesis\data\r...,quota_2025_aoi35.csv
6,1,20250913_061708_13_2547,76.869,100.000,NaN,D:\Thesis\glacial_lake_detection_thesis\data\r...,quota_2025_aoi35.csv
7,1,20250913_061705_81_2547,370.463,370.463,NaN,D:\Thesis\glacial_lake_detection_thesis\data\r...,quota_2025_aoi35.csv
8,1,20250927_061732_84_253c,384.391,384.391,NaN,D:\Thesis\glacial_lake_detection_thesis\data\r...,quota_2025_aoi36.csv
9,1,20250927_061735_13_253c,57.023,100.000,NaN,D:\Thesis\glacial_lake_detection_thesis\data\r...,quota_2025_aoi36.csv
10,1,20250927_061859_72_24fe,55.607,100.000,NaN,D:\Thesis\glacial_lake_detection_thesis\data\r...,quota_2025_aoi36.csv
11,1,20250927_061902_03_24fe,73.171,100.000,NaN,D:\Thesis\glacial_lake_detection_thesis\data\r...,quota_2025_aoi36.csv
12,1,20250902_062415_40_24f8,111.302,111.302,NaN,D:\Thesis\glacial_lake_detection_thesis\data\r...,quota_2025_aoi37.csv
13,1,20250902_062413_34_24f8,42.727,100.000,NaN,D:\Thesis\glacial_lake_detection_thesis\data\r...,quota_2025_aoi37.csv
14,1,20250929_062148_37_24b8,71.322,100.000,NaN,D:\Thesis\glacial_lake_detection_thesis\data\r...,quota_2025_aoi38.csv


In [8]:
for idx, row in order_df.iterrows():
    print(row["scene_id"])


20250913_061703_50_2547
20250913_061708_13_2547
20250913_061705_81_2547
20250927_061732_84_253c
20250927_061735_13_253c
20250927_061859_72_24fe
20250927_061902_03_24fe
20250902_062415_40_24f8
20250902_062413_34_24f8
20250929_062148_37_24b8
20250911_061934_74_24dc
20250911_061936_91_24dc
20250911_062301_64_2533
20250911_062303_97_2533
20250911_062306_30_2533
20250911_061939_09_24dc
20250916_062858_12_2508
20250916_062900_26_2508
20250916_062902_40_2508
20250916_062904_55_2508


In [9]:
for idx, row in order_df.iterrows():
    aoi_geom = read_aoi_geojson(Path(row["aoi_path"]))

    tools = [
        order_request.clip_tool(aoi_geom),
        order_request.reproject_tool(REPROJECT_EPSG),
        order_request.file_format_tool("COG"),
    ]

    delivery_cfg = None
    if ZIP_EACH_ORDER:
        delivery_cfg = order_request.delivery(archive_type="zip", single_archive=True)

    order_ids = []
    
    order_name = f"PSScene_{YEAR}_AOI_{idx:02d}_delta"
    print(f"Creating order with scene ID {row["scene_id"]}")

    req = order_request.build_request(
        name=order_name,
        products=[order_request.product(item_ids=[row["scene_id"]], product_bundle=PRIMARY_BUNDLE, item_type="PSScene",
                                            fallback_bundle=FALLBACK_BUNDLES or None)],
        tools=tools,
        delivery=delivery_cfg
    )
    order = pl.orders.create_order(req)
    oid = order["id"]
    order_ids.append(oid)
    print("  created:", oid)

    with reporting.StateBar(state="creating") as bar:
        pl.orders.wait(oid, callback=bar.update_state, max_attempts=WAIT_MAX_ATTEMPTS)

    order_dir = OUT_DIR / f"order_{idx:02d}_{oid}"
    order_dir.mkdir(parents=True, exist_ok=True)
    print("  downloading to:", order_dir)
    pl.orders.download_order(oid, directory=str(order_dir), overwrite=True)

    order_ids


Creating order with scene ID 20250913_061703_50_2547
  created: 64fd889d-3254-4633-be6f-e1a495e84713


15:15 - order  - state: success 


  downloading to: D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\inference\planet_downloads_PSScene_rgb_2025\order_05_64fd889d-3254-4633-be6f-e1a495e84713
Creating order with scene ID 20250913_061708_13_2547
  created: b59bfc5f-e766-4f9f-937f-f31d7d700dcd


12:16 - order  - state: success 


  downloading to: D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\inference\planet_downloads_PSScene_rgb_2025\order_06_b59bfc5f-e766-4f9f-937f-f31d7d700dcd
Creating order with scene ID 20250913_061705_81_2547
  created: cfc2088c-70b4-4d2c-8ee7-cedb0fd19c26


...

  downloading to: D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\inference\planet_downloads_PSScene_rgb_2025\order_22_1cc4ba95-5786-4495-b97c-0c5abc7fc101
Creating order with scene ID 20250916_062902_40_2508
  created: 5907223c-3eb9-4aa6-8dcd-02e80fa802e2


18:02 - order  - state: success 


  downloading to: D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\inference\planet_downloads_PSScene_rgb_2025\order_23_5907223c-3eb9-4aa6-8dcd-02e80fa802e2
Creating order with scene ID 20250916_062904_55_2508
  created: 434fc64b-74d6-4e44-9afe-c5439bacbb49


18:22 - order  - state: success 


  downloading to: D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\inference\planet_downloads_PSScene_rgb_2025\order_24_434fc64b-74d6-4e44-9afe-c5439bacbb49
